# Deep Belief Network in JAX

## Training: MCMC Distribution Evolution vs Diffusion Models

**Key insight**: In RBMs/DBNs, distribution evolution (Gibbs sampling) happens *during training* as part of the gradient estimate. In diffusion models, iterative refinement only happens at inference.

| | RBM/DBN | Diffusion Model |
|---|---|---|
| **Training** | MCMC chains (CD-k Gibbs steps) to estimate gradient | Direct: sample (t, noise), predict noise |
| **Inference** | MCMC sampling from learned distribution | Iterative denoising (many steps) |
| **One-step?** | No — need MCMC to sample | Yes — consistency/flow models |
| **What's learned** | Energy function E(v,h) | Score/noise predictor |
| **Distribution** | Boltzmann: p(v) ∝ Σ_h exp(-E(v,h)) | Learned reverse SDE/ODE |

In [ ]:
# Install dependencies
!pip install -q jax jaxlib tensorflow-datasets matplotlib

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import matplotlib.pyplot as plt

print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

## Part 1: Restricted Boltzmann Machine (RBM)

The building block. An RBM defines an energy:

$$E(v, h) = -v^T W h - b^T v - c^T h$$

The joint probability is $p(v, h) = \frac{1}{Z} \exp(-E(v,h))$ where $Z$ is the intractable partition function.

Training uses **Contrastive Divergence**: run k steps of Gibbs sampling to approximate the negative phase gradient. This is the *distribution evolution during training*.

In [ ]:
from rbm import (
    RBMParams, init_rbm, cd_k, init_train_state, update_params,
    visible_to_hidden_prob, hidden_to_visible_prob,
    generate_samples, compute_free_energy, sample_binary, gibbs_step,
)
from train_rbm import load_mnist_binary, make_batches

# Load binarized MNIST
train_data, test_data, train_labels, test_labels = load_mnist_binary()
print(f"Train: {train_data.shape}, Test: {test_data.shape}")

# Show some examples
fig, axes = plt.subplots(1, 10, figsize=(15, 1.5))
for i in range(10):
    axes[i].imshow(train_data[i].reshape(28, 28), cmap='gray')
    axes[i].axis('off')
plt.suptitle('Binarized MNIST samples')
plt.show()

### Train RBM with CD-1

In [ ]:
from train_rbm import train_rbm, plot_filters, plot_reconstructions, plot_samples, plot_training_curves

# Train the RBM
params, history, datasets = train_rbm(
    n_hidden=500,
    n_epochs=30,
    cd_steps=1,  # CD-1: one Gibbs step per gradient update
    lr=0.01,
    batch_size=64,
)

In [ ]:
# Training curves
plot_training_curves(history)

In [ ]:
# Learned filters — should see stroke/edge-like features
plot_filters(params, n_filters=100)

In [ ]:
# Reconstructions (v -> h -> v')
plot_reconstructions(params, test_data)

In [ ]:
# Generated samples — pure MCMC from the model
plot_samples(params, random.PRNGKey(42), n_samples=100, n_gibbs=2000)

### Visualize the Gibbs Chain (Distribution Evolution)

This shows the MCMC chain evolving — starting from noise and gradually settling into the model's learned distribution. This is the same process that happens during CD-k training (but there we only run k steps).

In [ ]:
# Visualize Gibbs chain evolution
key = random.PRNGKey(123)
v = random.bernoulli(key, 0.5, (1, 784)).astype(jnp.float32)  # start from noise

snapshots = []
snapshot_steps = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000]

for step in range(max(snapshot_steps) + 1):
    if step in snapshot_steps:
        h_p = visible_to_hidden_prob(params, v)
        v_p = hidden_to_visible_prob(params, h_p)
        snapshots.append((step, np.array(v_p[0]).reshape(28, 28)))
    key, k1, k2 = random.split(key, 3)
    h_prob = visible_to_hidden_prob(params, v)
    h_sample = sample_binary(k1, h_prob)
    v_prob = hidden_to_visible_prob(params, h_sample)
    v = sample_binary(k2, v_prob)

fig, axes = plt.subplots(1, len(snapshots), figsize=(2 * len(snapshots), 2.5))
for i, (step, img) in enumerate(snapshots):
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f't={step}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Gibbs Chain: Noise → Learned Distribution', fontsize=13)
plt.tight_layout()
plt.show()

## Part 2: Deep Belief Network

Stack RBMs and train greedily layer by layer. Each layer learns more abstract features.

**DBN Architecture**: 784 → 500 → 200 → 50

After pretraining, we can:
1. Generate samples (top-down through layers)
2. Fine-tune as a classifier (add softmax, backprop)

In [ ]:
from dbn import (
    greedy_layerwise_pretrain, dbn_generate,
    propagate_up, propagate_down,
    finetune_classifier,
)

# Greedy layer-wise pretraining
dbn_params = greedy_layerwise_pretrain(
    data=train_data,
    layer_sizes=[784, 500, 200, 50],
    n_epochs_per_layer=30,
    cd_steps=1,
    batch_size=64,
    lr=0.01,
)

In [ ]:
# Visualize filters from each layer
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for layer_idx in range(3):
    W = np.array(dbn_params.rbm_layers[layer_idx].W)
    for j in range(10):
        if layer_idx == 0:
            # First layer filters are in pixel space
            img = W[:, j].reshape(28, 28)
        else:
            # Higher layer filters need to be projected back
            h = np.zeros(W.shape[1])
            h[j] = 1.0
            v = np.array(propagate_down(dbn_params, jnp.array(h.reshape(1, -1)), from_layer=layer_idx))[0]
            img = v.reshape(28, 28)
        axes[layer_idx, j].imshow(img, cmap='gray' if layer_idx > 0 else 'RdBu_r')
        axes[layer_idx, j].axis('off')
    axes[layer_idx, 0].set_ylabel(f'Layer {layer_idx+1}', fontsize=11, rotation=0, labelpad=40)
plt.suptitle('DBN Features at Each Layer', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Generate samples from the DBN
print("Generating samples from DBN (Gibbs in top RBM, then propagate down)...")
samples = dbn_generate(dbn_params, random.PRNGKey(0), n_samples=100, n_gibbs=2000)

fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for i in range(100):
    axes[i // 10, i % 10].imshow(np.array(samples[i]).reshape(28, 28), cmap='gray')
    axes[i // 10, i % 10].axis('off')
plt.suptitle('DBN Generated Samples', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# DBN reconstructions through full stack
n_show = 10
v_input = jnp.array(test_data[:n_show])

# Encode all the way up, then decode all the way down
h_top = propagate_up(dbn_params, v_input)
v_recon = propagate_down(dbn_params, h_top)

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 1.5, 3))
for i in range(n_show):
    axes[0, i].imshow(np.array(v_input[i]).reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(np.array(v_recon[i]).reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_title('Original', fontsize=10)
axes[1, 0].set_title('DBN Recon', fontsize=10)
plt.suptitle('DBN Reconstructions (784 → 500 → 200 → 50 → 200 → 500 → 784)', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3: Fine-tune DBN as Classifier

This was the original use case (Hinton & Salakhutdinov, 2006):
1. Pretrain with unsupervised CD (learns good features)
2. Add softmax layer
3. Fine-tune end-to-end with backprop

The pretraining helps avoid bad local minima and provides a good initialization for deep networks.

In [ ]:
# Fine-tune as a digit classifier
clf_params = finetune_classifier(
    dbn_params,
    train_data, train_labels,
    test_data, test_labels,
    n_classes=10,
    n_epochs=30,
    batch_size=64,
    lr=0.001,
)

## Part 4: Comparing Training Dynamics

Let's visualize what CD-k looks like for different k values. Higher k = longer Gibbs chain = better gradient estimate but slower training.

In [ ]:
# Compare CD-1, CD-5, CD-10 on a small RBM
histories = {}
for k in [1, 5, 10]:
    print(f"\n--- Training with CD-{k} ---")
    p, h, _ = train_rbm(n_hidden=200, n_epochs=20, cd_steps=k, batch_size=128)
    histories[f'CD-{k}'] = h['recon_error']

plt.figure(figsize=(8, 5))
for name, errors in histories.items():
    plt.plot(errors, label=name)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Error')
plt.title('Effect of Gibbs Steps (k) in CD-k Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary: DBN vs Diffusion Training

### RBM/DBN Training
- **Gradient computation** needs MCMC (Gibbs sampling = distribution evolution)
- `∇log p(v) = <vh^T>_data - <vh^T>_model` — second term requires sampling from the model
- CD-k approximates this with k Gibbs steps (short chain, biased but effective)
- PCD maintains persistent chains across updates for better mixing

### Diffusion Training
- **No MCMC during training**: sample timestep t ~ U[0,T], noise ε ~ N(0,I)
- Closed-form forward process: `x_t = √ᾱ_t x_0 + √(1-ᾱ_t) ε`
- Loss: `||ε - ε_θ(x_t, t)||²` — simple regression
- The iterative refinement is deferred entirely to inference

### The Philosophical Connection
Both are learning energy landscapes, but:
- RBM explicitly parameterizes E(v,h) and samples from the Boltzmann distribution
- Diffusion models implicitly define an energy via the score function ∇log p(x)
- In both, the model's "belief" about the data is a distribution that can be sampled from
- The key difference is *when* and *how* that sampling happens